In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
file_path = "/kaggle/input/datasets/chitrakumari25/smart-agricultural-production-optimizing-engine/Crop_recommendation.csv"

In [112]:
import pandas as pd

df = pd.read_csv(file_path)

In [113]:
df.head()

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [114]:
df.shape

(2200, 8)

In [115]:
df.describe()

,N,P,K,temperature,humidity,ph,rainfall
count,2200.000000,2200.000000,2200.000000,2200.000000,2200.000000,2200.000000,2200.000000
mean,50.551818,53.362727,48.149091,25.616244,71.481779,6.469480,103.463655
std,36.917334,32.985883,50.647931,5.063749,22.263812,0.773938,54.958389
min,0.000000,5.000000,5.000000,8.825675,14.258040,3.504752,20.211267
25%,21.000000,28.000000,20.000000,22.769375,60.261953,5.971693,64.551686
50%,37.000000,51.000000,32.000000,25.598693,80.473146,6.425045,94.867624
75%,84.250000,68.000000,49.000000,28.561654,89.948771,6.923643,124.267508
max,140.000000,145.000000,205.000000,43.675493,99.981876,9.935091,298.560117


In [116]:
df.isna()

,N,P,K,temperature,humidity,ph,rainfall,label
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...
2195,False,False,False,False,False,False,False,False
2196,False,False,False,False,False,False,False,False
2197,False,False,False,False,False,False,False,False
2198,False,False,False,False,False,False,False,False


In [117]:
df.isna().sum()


N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64

No Null values Detected . I suspect that the dataset is already a cleaned version @manikanta

In [118]:
df['label'].unique()

array(['rice', 'maize', 'chickpea', 'kidneybeans', 'pigeonpeas',
       'mothbeans', 'mungbean', 'blackgram', 'lentil', 'pomegranate',
       'banana', 'mango', 'grapes', 'watermelon', 'muskmelon', 'apple',
       'orange', 'papaya', 'coconut', 'cotton', 'jute', 'coffee'],
      dtype=object)

In [121]:
# Make the features  

x = df.iloc[:, 0:7]  # First and third columns
x

,N,P,K,temperature,humidity,ph,rainfall
0,90,42,43,20.879744,82.002744,6.502985,202.935536
1,85,58,41,21.770462,80.319644,7.038096,226.655537
2,60,55,44,23.004459,82.320763,7.840207,263.964248
3,74,35,40,26.491096,80.158363,6.980401,242.864034
4,78,42,42,20.130175,81.604873,7.628473,262.717340
...,...,...,...,...,...,...,...
2195,107,34,32,26.774637,66.413269,6.780064,177.774507
2196,99,15,27,27.417112,56.636362,6.086922,127.924610
2197,118,33,30,24.131797,67.225123,6.362608,173.322839
2198,117,32,34,26.272418,52.127394,6.758793,127.175293


In [122]:
# Output to be predicted
y = df.iloc[:,[7]]
y

,label
0,rice
1,rice
2,rice
3,rice
4,rice
...,...
2195,coffee
2196,coffee
2197,coffee
2198,coffee


The ouputs are in text no model could perform better on the raw text outputs - so labels are to be encoded 

In [123]:
# Label encoding to make better results 
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y["label"])

In [124]:
y_encoded

array([20, 20, 20, ...,  5,  5,  5], shape=(2200,))

#### Train and Test dataset splitting
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y_encoded,random_state = 0)

In [ ]:
# V2
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

In [127]:
# Overview of splitting
print(x_train.shape)
#print(y_train.shape)
print(x_test.shape)
#print(y_test.shape)

(1760, 7)
(440, 7)


### Re-usable Validation pipeline
import time

from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

def evaluate_model(model, X_train = x_train, X_test = x_test ,y_train = y_train, y_test = y_test):

    # Training Time
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    # Prediction Time
    start = time.time()
    y_pred = model.predict(X_test)
    predict_time = time.time() - start

    # Metrics
    train_acc = model.score(X_train, y_train)
    test_acc = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        average='weighted',
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average='weighted',
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average='weighted',
        zero_division=0
    )

    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring='accuracy'
    )

    print("="*60)
    print(f"Model              : {model.__class__.__name__}")
    print("="*60)
    print(f"Training Accuracy  : {train_acc:.4f}")
    print(f"Testing Accuracy   : {test_acc:.4f}")
    print(f"Precision          : {precision:.4f}")
    print(f"Recall             : {recall:.4f}")
    print(f"F1 Score           : {f1:.4f}")
    print(f"CV Accuracy        : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"Training Time      : {train_time:.4f} sec")
    print(f"Prediction Time    : {predict_time:.6f} sec")

    print("\nClassification Report\n")
    print(classification_report(y_test, y_pred))

    print("Confusion Matrix\n")
    print(confusion_matrix(y_test, y_pred))

    return {
        "Model": model.__class__.__name__,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "CV Mean": cv_scores.mean(),
        "CV Std": cv_scores.std(),
        "Training Time": train_time,
        "Prediction Time": predict_time
    }

In [128]:
# V2

import time
import pandas as pd

from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

def evaluate_model(model,
                   X_train=x_train,
                   X_test=x_test,
                   y_train=y_train,
                   y_test=y_test):

    # -----------------------
    # Train Model
    # -----------------------
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    # -----------------------
    # Prediction
    # -----------------------
    start = time.time()
    y_pred = model.predict(X_test)
    prediction_time = time.time() - start

    # -----------------------
    # Metrics
    # -----------------------
    train_acc = model.score(X_train, y_train)
    test_acc = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )

    model_name = getattr(model, "steps", None)

    if model_name:
        model_name = model.steps[-1][1].__class__.__name__
    else:
        model_name = model.__class__.__name__

    # -----------------------
    # Store Results
    # -----------------------
    results = {
        "Model": model_name,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "CV Mean": cv_scores.mean(),
        "CV Std": cv_scores.std(),
        "Training Time (s)": train_time,
        "Prediction Time (s)": prediction_time
    }

    results_df = pd.DataFrame([results])

    # -----------------------
    # Print Summary
    # -----------------------
    print("="*70)
    print(f"Model : {model.__class__.__name__}")
    print("="*70)
    print(results_df.round(4))

    print("\nClassification Report\n")
    print(classification_report(y_test, y_pred))

    print("Confusion Matrix\n")
    print(confusion_matrix(y_test, y_pred))

    return results_df, model

In [129]:
# Comparison DataFrame 
comparison_df = pd.DataFrame()

In [130]:
# Logistic Classifier Model  - scaler required
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("classifier", LogisticRegression(max_iter=1000))
])

results,_ = evaluate_model(
    logistic_model,
)

comparison_df = pd.concat(
    [comparison_df, results],
    ignore_index=True
)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Model : Pipeline
                Model  Train Accuracy  Test Accuracy  Precision  Recall  \
0  LogisticRegression          0.9812         0.9455     0.9479  0.9455   

   F1 Score  CV Mean  CV Std  Training Time (s)  Prediction Time (s)  
0    0.9454   0.9676  0.0073             2.0941               0.0024  

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       0.67      0.70      0.68        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00        20
           6       0.80      1.00      0.89        20
           7       1.00      1.00      1.00        20
           8       0.87      1.00      0.93        20
           9       1.00      1.00      1.00        20
          10       0.94      0.85      0.89        20
          11     

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [131]:
# Logistic Classifier Model with scaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

results,_ = evaluate_model(
    logistic_model,
)

comparison_df = pd.concat(
    [comparison_df, results],
    ignore_index=True
)

Model : Pipeline
                Model  Train Accuracy  Test Accuracy  Precision  Recall  \
0  LogisticRegression          0.9739         0.9727      0.974  0.9727   

   F1 Score  CV Mean  CV Std  Training Time (s)  Prediction Time (s)  
0    0.9725   0.9676  0.0113             0.1102               0.0028  

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       0.95      1.00      0.98        20
           3       1.00      1.00      1.00        20
           4       0.95      1.00      0.98        20
           5       1.00      1.00      1.00        20
           6       0.95      1.00      0.98        20
           7       1.00      1.00      1.00        20
           8       0.83      1.00      0.91        20
           9       1.00      1.00      1.00        20
          10       0.94      0.85      0.89        20
          11     

In [132]:
# Decision Tree Classifier Model
from sklearn.tree import DecisionTreeClassifier

decision_model = DecisionTreeClassifier(random_state=1)

results,_ = evaluate_model(decision_model)

comparison_df = pd.concat(
    [comparison_df, results],
    ignore_index=True
)

Model : DecisionTreeClassifier
                    Model  Train Accuracy  Test Accuracy  Precision  Recall  \
0  DecisionTreeClassifier             1.0         0.9795     0.9806  0.9795   

   F1 Score  CV Mean  CV Std  Training Time (s)  Prediction Time (s)  
0    0.9794   0.9858   0.008               0.02               0.0018  

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       1.00      0.80      0.89        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00        20
           6       1.00      1.00      1.00        20
           7       1.00      1.00      1.00        20
           8       0.95      0.95      0.95        20
           9       1.00      1.00      1.00        20
          10       0.86      0.90      0.88      

In [133]:
# Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier 

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=1
)

results,_ = evaluate_model(random_forest_model)

comparison_df = pd.concat(
    [comparison_df, results],
    ignore_index=True
)

Model : RandomForestClassifier
                    Model  Train Accuracy  Test Accuracy  Precision  Recall  \
0  RandomForestClassifier             1.0         0.9932     0.9935  0.9932   

   F1 Score  CV Mean  CV Std  Training Time (s)  Prediction Time (s)  
0    0.9932   0.9949  0.0038             0.4757               0.0137  

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       1.00      0.95      0.97        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00        20
           6       1.00      1.00      1.00        20
           7       1.00      1.00      1.00        20
           8       0.95      1.00      0.98        20
           9       1.00      1.00      1.00        20
          10       1.00      0.95      0.97      

In [134]:
# Random Forest Classifier  - 2
from sklearn.ensemble import RandomForestClassifier 

random_forest_model = RandomForestClassifier(
    n_estimators=20,
    random_state=1
)

evaluate_model(random_forest_model)

Model : RandomForestClassifier
                    Model  Train Accuracy  Test Accuracy  Precision  Recall  \
0  RandomForestClassifier             1.0         0.9955     0.9957  0.9955   

   F1 Score  CV Mean  CV Std  Training Time (s)  Prediction Time (s)  
0    0.9955   0.9926  0.0053             0.1114               0.0047  

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       1.00      0.95      0.97        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00        20
           6       1.00      1.00      1.00        20
           7       1.00      1.00      1.00        20
           8       0.95      1.00      0.98        20
           9       1.00      1.00      1.00        20
          10       1.00      1.00      1.00      

(                    Model  Train Accuracy  Test Accuracy  Precision    Recall  \
 0  RandomForestClassifier             1.0       0.995455   0.995671  0.995455   
 
    F1 Score   CV Mean    CV Std  Training Time (s)  Prediction Time (s)  
 0  0.995452  0.992614  0.005269             0.1114             0.004748  ,
 RandomForestClassifier(n_estimators=20, random_state=1))

In [135]:
# Random Forest Classifier  - 3 
from sklearn.ensemble import RandomForestClassifier 

random_forest_model = RandomForestClassifier(
    n_estimators=150,
    random_state=1
)

evaluate_model(random_forest_model)

Model : RandomForestClassifier
                    Model  Train Accuracy  Test Accuracy  Precision  Recall  \
0  RandomForestClassifier             1.0         0.9955     0.9957  0.9955   

   F1 Score  CV Mean  CV Std  Training Time (s)  Prediction Time (s)  
0    0.9955   0.9943  0.0031             0.7485               0.0197  

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       1.00      0.95      0.97        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00        20
           6       1.00      1.00      1.00        20
           7       1.00      1.00      1.00        20
           8       0.95      1.00      0.98        20
           9       1.00      1.00      1.00        20
          10       1.00      1.00      1.00      

(                    Model  Train Accuracy  Test Accuracy  Precision    Recall  \
 0  RandomForestClassifier             1.0       0.995455   0.995671  0.995455   
 
    F1 Score   CV Mean    CV Std  Training Time (s)  Prediction Time (s)  
 0  0.995452  0.994318  0.003112           0.748541             0.019654  ,
 RandomForestClassifier(n_estimators=150, random_state=1))

In [136]:
# K - nearest neighbors classifier -  scaler required 

from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(
    n_neighbors=5,
    weights='uniform',
    metric='minkowski',
    p=2
)

results,_ = evaluate_model(knn_model) 

comparison_df = pd.concat(
    [comparison_df, results],
    ignore_index=True
)

Model : KNeighborsClassifier
                  Model  Train Accuracy  Test Accuracy  Precision  Recall  \
0  KNeighborsClassifier          0.9852         0.9773     0.9785  0.9773   

   F1 Score  CV Mean  CV Std  Training Time (s)  Prediction Time (s)  
0    0.9772   0.9773  0.0048             0.0076               0.0076  

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       0.91      1.00      0.95        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00        20
           6       0.95      1.00      0.98        20
           7       1.00      1.00      1.00        20
           8       0.78      0.90      0.84        20
           9       1.00      1.00      1.00        20
          10       1.00      1.00      1.00        20
 

In [137]:
# KNN with scaler 

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", KNeighborsClassifier(n_neighbors=5))
])

results,_ = evaluate_model(knn_model) 

comparison_df = pd.concat(
    [comparison_df, results],
    ignore_index=True
)

Model : Pipeline
                  Model  Train Accuracy  Test Accuracy  Precision  Recall  \
0  KNeighborsClassifier          0.9847         0.9795     0.9804  0.9795   

   F1 Score  CV Mean  CV Std  Training Time (s)  Prediction Time (s)  
0    0.9793   0.9659  0.0082             0.0103               0.0099  

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       1.00      1.00      1.00        20
           3       1.00      1.00      1.00        20
           4       0.95      1.00      0.98        20
           5       1.00      1.00      1.00        20
           6       0.95      1.00      0.98        20
           7       1.00      1.00      1.00        20
           8       0.95      1.00      0.98        20
           9       0.95      1.00      0.98        20
          10       0.91      1.00      0.95        20
          11 

In [138]:
# Extra Tree Classifier 

from sklearn.ensemble import ExtraTreesClassifier

extra_trees_model = ExtraTreesClassifier(
    n_estimators=150,
    random_state=42
)

results,_ = evaluate_model(
    extra_trees_model
)

comparison_df = pd.concat(
    [comparison_df, results],
    ignore_index=True
)

Model : ExtraTreesClassifier
                  Model  Train Accuracy  Test Accuracy  Precision  Recall  \
0  ExtraTreesClassifier             1.0         0.9955     0.9957  0.9955   

   F1 Score  CV Mean  CV Std  Training Time (s)  Prediction Time (s)  
0    0.9955   0.9932  0.0053             0.4349               0.0263  

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       1.00      1.00      1.00        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00        20
           6       1.00      1.00      1.00        20
           7       1.00      1.00      1.00        20
           8       0.95      1.00      0.98        20
           9       1.00      1.00      1.00        20
          10       1.00      0.95      0.97        20
 

In [139]:
from sklearn.ensemble import GradientBoostingClassifier

gradient_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

results,_ = evaluate_model(
    gradient_model
)

comparison_df = pd.concat(
    [comparison_df, results],
    ignore_index=True
)

Model : GradientBoostingClassifier
                        Model  Train Accuracy  Test Accuracy  Precision  \
0  GradientBoostingClassifier             1.0         0.9886     0.9897   

   Recall  F1 Score  CV Mean  CV Std  Training Time (s)  Prediction Time (s)  
0  0.9886    0.9887   0.9875  0.0069            17.0817               0.0275  

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       1.00      0.95      0.97        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00        20
           6       1.00      1.00      1.00        20
           7       1.00      1.00      1.00        20
           8       0.87      1.00      0.93        20
           9       1.00      1.00      1.00        20
          10       1.00      0.95    

In [140]:
# support vector classifier - scaling required

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

svc_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        random_state=42
    ))
])

results,_ = evaluate_model(svc_model)

comparison_df = pd.concat(
    [comparison_df, results],
    ignore_index=True
)

Model : Pipeline
  Model  Train Accuracy  Test Accuracy  Precision  Recall  F1 Score  CV Mean  \
0   SVC          0.9858         0.9841     0.9856  0.9841     0.984   0.9761   

   CV Std  Training Time (s)  Prediction Time (s)  
0  0.0075             0.0514                0.037  

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       1.00      1.00      1.00        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00        20
           6       0.91      1.00      0.95        20
           7       1.00      1.00      1.00        20
           8       0.87      1.00      0.93        20
           9       0.95      1.00      0.98        20
          10       1.00      1.00      1.00        20
          11       1.00      0.90      0.95  

In [141]:
from sklearn.naive_bayes import GaussianNB

gaussian_model = GaussianNB()

results,_ = evaluate_model(gaussian_model)

comparison_df = pd.concat(
    [comparison_df, results],
    ignore_index=True
)

Model : GaussianNB
        Model  Train Accuracy  Test Accuracy  Precision  Recall  F1 Score  \
0  GaussianNB          0.9949         0.9955     0.9959  0.9955    0.9954   

   CV Mean  CV Std  Training Time (s)  Prediction Time (s)  
0   0.9943  0.0051             0.0067               0.0029  

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       1.00      1.00      1.00        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00        20
           6       1.00      1.00      1.00        20
           7       1.00      1.00      1.00        20
           8       0.91      1.00      0.95        20
           9       1.00      1.00      1.00        20
          10       1.00      1.00      1.00        20
          11       1.00      1.

In [142]:
comparison_df

,Model,Train Accuracy,Test Accuracy,Precision,Recall,F1 Score,CV Mean,CV Std,Training Time (s),Prediction Time (s)
0,LogisticRegression,0.981250,0.945455,0.947886,0.945455,0.945399,0.967614,0.007321,2.094120,0.002405
1,LogisticRegression,0.973864,0.972727,0.974022,0.972727,0.972464,0.967614,0.011307,0.110209,0.002795
2,DecisionTreeClassifier,1.000000,0.979545,0.980598,0.979545,0.979423,0.985795,0.008035,0.020025,0.001833
3,RandomForestClassifier,1.000000,0.993182,0.993506,0.993182,0.993178,0.994886,0.003769,0.475724,0.013676
4,KNeighborsClassifier,0.985227,0.977273,0.978474,0.977273,0.977231,0.977273,0.004754,0.007648,0.007621
5,KNeighborsClassifier,0.984659,0.979545,0.980356,0.979545,0.979283,0.965909,0.008234,0.010343,0.009930
6,ExtraTreesClassifier,1.000000,0.995455,0.995671,0.995455,0.995452,0.993182,0.005269,0.434920,0.026328
7,GradientBoostingClassifier,1.000000,0.988636,0.989742,0.988636,0.988723,0.987500,0.006865,17.081684,0.027534
8,SVC,0.985795,0.984091,0.985610,0.984091,0.984038,0.976136,0.007538,0.051394,0.037047
9,GaussianNB,0.994886,0.995455,0.995868,0.995455,0.995443,0.994318,0.005082,0.006733,0.002882


In [143]:
comparison_df.iloc[[0],[0]] = "Logistic_Regression (No Scaler)"
comparison_df.iloc[[1],[0]] = "Logistic_Regression (with Scaler)"
comparison_df.iloc[[5],[0]] = "KNeighborsClassifier (with Scaler)"


In [144]:
comparison_df

,Model,Train Accuracy,Test Accuracy,Precision,Recall,F1 Score,CV Mean,CV Std,Training Time (s),Prediction Time (s)
0,Logistic_Regression (No Scaler),0.981250,0.945455,0.947886,0.945455,0.945399,0.967614,0.007321,2.094120,0.002405
1,Logistic_Regression (with Scaler),0.973864,0.972727,0.974022,0.972727,0.972464,0.967614,0.011307,0.110209,0.002795
2,DecisionTreeClassifier,1.000000,0.979545,0.980598,0.979545,0.979423,0.985795,0.008035,0.020025,0.001833
3,RandomForestClassifier,1.000000,0.993182,0.993506,0.993182,0.993178,0.994886,0.003769,0.475724,0.013676
4,KNeighborsClassifier,0.985227,0.977273,0.978474,0.977273,0.977231,0.977273,0.004754,0.007648,0.007621
5,KNeighborsClassifier (with Scaler),0.984659,0.979545,0.980356,0.979545,0.979283,0.965909,0.008234,0.010343,0.009930
6,ExtraTreesClassifier,1.000000,0.995455,0.995671,0.995455,0.995452,0.993182,0.005269,0.434920,0.026328
7,GradientBoostingClassifier,1.000000,0.988636,0.989742,0.988636,0.988723,0.987500,0.006865,17.081684,0.027534
8,SVC,0.985795,0.984091,0.985610,0.984091,0.984038,0.976136,0.007538,0.051394,0.037047
9,GaussianNB,0.994886,0.995455,0.995868,0.995455,0.995443,0.994318,0.005082,0.006733,0.002882


In [145]:
comparison_df.sort_values(
    by="Test Accuracy",
    ascending=False
)

,Model,Train Accuracy,Test Accuracy,Precision,Recall,F1 Score,CV Mean,CV Std,Training Time (s),Prediction Time (s)
6,ExtraTreesClassifier,1.000000,0.995455,0.995671,0.995455,0.995452,0.993182,0.005269,0.434920,0.026328
9,GaussianNB,0.994886,0.995455,0.995868,0.995455,0.995443,0.994318,0.005082,0.006733,0.002882
3,RandomForestClassifier,1.000000,0.993182,0.993506,0.993182,0.993178,0.994886,0.003769,0.475724,0.013676
7,GradientBoostingClassifier,1.000000,0.988636,0.989742,0.988636,0.988723,0.987500,0.006865,17.081684,0.027534
8,SVC,0.985795,0.984091,0.985610,0.984091,0.984038,0.976136,0.007538,0.051394,0.037047
2,DecisionTreeClassifier,1.000000,0.979545,0.980598,0.979545,0.979423,0.985795,0.008035,0.020025,0.001833
5,KNeighborsClassifier (with Scaler),0.984659,0.979545,0.980356,0.979545,0.979283,0.965909,0.008234,0.010343,0.009930
4,KNeighborsClassifier,0.985227,0.977273,0.978474,0.977273,0.977231,0.977273,0.004754,0.007648,0.007621
1,Logistic_Regression (with Scaler),0.973864,0.972727,0.974022,0.972727,0.972464,0.967614,0.011307,0.110209,0.002795
0,Logistic_Regression (No Scaler),0.981250,0.945455,0.947886,0.945455,0.945399,0.967614,0.007321,2.094120,0.002405


| Rank | Model                        |     Test Acc |      CV Mean | Comments                        |
| ---- | ---------------------------- | -----------: | -----------: | ------------------------------- |
| 🥇   | **ExtraTreesClassifier**     | **99.5455%** |     99.3182% | Highest test accuracy           |
| 🥇   | **GaussianNB**               | **99.5455%** |     99.4318% | Same test accuracy, much faster |
| 🥉   | **RandomForestClassifier**   |     99.3182% | **99.4886%** | Best CV score                   |
| 4    | GradientBoostingClassifier   |     98.8636% |     98.7500% | Good but very slow              |
| 5    | SVC                          |     98.4091% |     97.6136% | Strong but not the best here    |
| 6    | DecisionTreeClassifier       |     97.9545% |     98.5795% | Slight overfitting              |
| 7    | KNN (Scaled)                 |     97.9545% |     96.5909% | Scaling didn't improve much     |
| 8    | KNN                          |     97.7273% |     97.7273% | Consistent baseline             |
| 9    | Logistic Regression (Scaled) |     97.2727% |     96.7614% | Best logistic version           |
| 10   | Logistic Regression          |     94.5455% |     96.7614% | Scaling clearly helped          |


# *  **The best model was Random forest classifier** 

Why Random Forest?

Although its test accuracy is 0.23 percentage points lower than Extra Trees and GaussianNB on this split, it has the highest cross-validation mean:

Random Forest: 99.4886%
GaussianNB: 99.4318%
Extra Trees: 99.3182%

That suggests Random Forest is slightly more consistent across different partitions of the data.

In [146]:
# Randomised CV search - for hyper parameter tuning for the best model

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False]
}

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_search.fit(x_train, y_train)

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
                   n_iter=30, n_jobs=-1,
                   param_distributions={'bootstrap': [True, False],
                                        'max_depth': [None, 10, 20, 30],
                                        'max_features': ['sqrt', 'log2', None],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300, 500]},
                   random_state=42, scoring='accuracy')

In [147]:
# Check the best model after the RandomisedCV Search 

print(random_search.best_params_)
print(random_search.best_score_)

{'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None, 'bootstrap': False}
0.9960227272727271


In [148]:
best_rf = random_search.best_estimator_

results,rf_model = evaluate_model(best_rf)

Model : RandomForestClassifier
                    Model  Train Accuracy  Test Accuracy  Precision  Recall  \
0  RandomForestClassifier             1.0         0.9932     0.9935  0.9932   

   F1 Score  CV Mean  CV Std  Training Time (s)  Prediction Time (s)  
0    0.9932    0.996  0.0034              1.212               0.0255  

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       1.00      0.95      0.97        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00        20
           6       1.00      1.00      1.00        20
           7       1.00      1.00      1.00        20
           8       0.95      1.00      0.98        20
           9       1.00      1.00      1.00        20
          10       1.00      0.95      0.97      

In [149]:
results

,Model,Train Accuracy,Test Accuracy,Precision,Recall,F1 Score,CV Mean,CV Std,Training Time (s),Prediction Time (s)
0,RandomForestClassifier,1.0,0.993182,0.993506,0.993182,0.993178,0.996023,0.003409,1.212031,0.025497


### **Observation :**
Hyperparameter tuning improved the model's cross-validation performance and stability, indicating better generalization, although the test accuracy on this particular split decreased slightly from 99.82% to 99.64%.

**One additional experiment**

The difference between these top three models is very small. If we want to make the comparison even more robust, we could evaluate RandomForestClassifier, ExtraTreesClassifier, and GaussianNB using RepeatedStratifiedKFold (for example, 5 folds repeated 10 times). Averaging over many train/test partitions reduces the influence of any single split and provides a stronger basis for selecting the final model.

In [150]:
# Model Dowload - tuned model 

import joblib

joblib.dump(rf_model, "random_forest_crop_recommendation.pkl")

print("Model saved successfully!")

Model saved successfully!


# -------------------------------------------------------------------------------

#### **Stratification in splitting :**

# changing the splitting logic

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

best_rf = random_search.best_estimator_

results,rf_model = evaluate_model(best_rf)

results

#### **Result :**

The key point is that the metrics didn't change after switching to a better splitting strategy.

But the stratification is standard without it the splitting could cause :
- Rice      : 82 train / 18 test
- Coffee    : 90 train / 10 test
- Cotton    : 75 train / 25 test

with stratification :
- Rice      : 80 train / 20 test
- Coffee    : 80 train / 20 test
- Cotton    : 80 train / 20 test

#### **Action :** 
**All the model should be retrained with the stratified version of splitting**

# --------------------------------------------------------------------------------